# Experiments 

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import mlflow
import shap


from sklearn.preprocessing import PowerTransformer, OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder, MinMaxScaler, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, SimpleImputer, IterativeImputer, MissingIndicator
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_validate
from sklearn.neighbors import LocalOutlierFactor

In [2]:
import dagshub
dagshub.init(repo_owner='bowlekarbhushan88', repo_name='property-price-prediction', mlflow=True)

Accessing as bowlekarbhushan88

Initialized MLflow to track repo "bowlekarbhushan88/property-price-prediction"

Repository bowlekarbhushan88/property-price-prediction initialized!

In [3]:
# set the tracking server

mlflow.set_tracking_uri("https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow")

In [4]:
#Here, I used the data saved using VS Code. This data has 184 columns, which means it is the result of merging df and am_df.
df = pd.read_csv(r'C:\Users\AMD\Desktop\bhushan PC data 11-8-2025/bhushan/property_project/files_vscode/data/py_cleaned_data.csv')

C:\Users\AMD\AppData\Local\Temp\ipykernel_98116\1989270230.py:2: DtypeWarning: Columns (106,128,134,135,136,137,138,139) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\AMD\Desktop\bhushan PC data 11-8-2025/bhushan/property_project/files_vscode/data/py_cleaned_data.csv')


In [5]:
# drop columns not required for model input 
columns_to_drop = ['id','price_category','costpersqft','emi']

df.drop(columns = columns_to_drop , inplace = True)

In [6]:
# combine all amenities and make one column
# Step 1: Filter columns that start with 'am_'
am_cols = [col for col in df.columns if col.startswith('am_')]

# Step 2: Combine values row-wise into a comma-separated string (not a list)
df['amenities'] = df[am_cols].apply(
    lambda row: ', '.join([str(val).strip() for val in row if isinstance(val, str) and val.strip() != ""]),
    axis=1
)

In [7]:
# Step 3: Drop original 'am_' columns
df.drop(columns=am_cols, inplace=True)

In [8]:
#drop duplicate rows 
df.drop_duplicates(inplace=True)

## Data preparation 

In [9]:
temp_df = df.copy()

In [10]:
# split data 

X = temp_df.drop(columns = 'price')
y = temp_df['price']

In [11]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (9302, 47)
The shape of test data is (2326, 47)


`lets categorized the columns`

## Transform Target column

In [13]:
pt = PowerTransformer(method='yeo-johnson')
y_train_trans = pd.Series(
    pt.fit_transform(y_train.values.reshape(-1, 1)).ravel(),
    index=y_train.index
)
y_test_trans = pd.Series(
    pt.transform(y_test.values.reshape(-1, 1)).ravel(),
    index=y_test.index
)

In [14]:
#o/p in daraframe
from sklearn import set_config
set_config(transform_output="pandas")

`Till this the code is common for all below experimants`

# experiment :06 (Baseline_model + permutation importance + standardization)

In [15]:
# mlflow experiment

mlflow.set_experiment("Exp 6 - Baseline model with permutation importance and standardization")

2025/08/18 11:38:43 INFO mlflow.tracking.fluent: Experiment with name 'Exp 6 - Baseline model and permutation importance' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/69e7c3392a3f43b9afd171045205836a', creation_time=1755497323861, experiment_id='8', last_update_time=1755497323861, lifecycle_stage='active', name='Exp 6 - Baseline model and permutation importance', tags={}>

In [16]:
# --- Custom MultiLabel Binarizer ---
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.columns = []

    def fit(self, X, y=None):
        self.columns = X.columns
        for col in self.columns:
            mlb = MultiLabelBinarizer()
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            mlb.fit(split_data)
            self.encoders[col] = mlb
        return self

    def transform(self, X):
        output_parts = []
        for col in self.columns:
            mlb = self.encoders[col]
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            transformed = mlb.transform(split_data)
            col_names = [f"{col}_{cls}" for cls in mlb.classes_]
            output_parts.append(pd.DataFrame(transformed, columns=col_names, index=X.index))
        return pd.concat(output_parts, axis=1)

In [17]:
# --- Custom Transformer: Top K Categories ---
class TopKCategoriesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=200):
        self.top_k = top_k
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            top = X[col].value_counts().nlargest(self.top_k).index
            self.top_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].where(X[col].isin(self.top_categories_[col]), other='__other__')
        return X

In [18]:
#final
amenities_weightages = {
    "sea facing": 10,
    "private pool": 10,
    "private jaccuzi": 10,
    "sky villa": 10,
    "helipad": 10,
    "wrap around balcony": 7,
    "infinity swimming pool": 10,
    "high ceiling": 9,
    "located in the heart of city": 10,
    "large open space": 10,
    "skyline view": 10,
    "private terrace/garden": 10,
    "private garage": 10,
    "mansion": 10,
    "club house": 9,
    "large clubhouse": 9,
    "modular kitchen": 9,
    "central ac": 9,
    "banquet hall": 6,
    "premium branded fittings": 9,
    "private garden": 9,
    "full glass wall": 9,
    "garden view": 9,
    "theme based architectures": 9,
    "grand entrance lobby": 9,
    "smart home": 9,
    "library and business centre": 9,
    "recreational pool": 9,
    "projector": 8,
    "swimming pool": 8,
    "gymnasium": 8,
    "indoor squash & badminton courts": 8,
    "outdoor tennis courts": 8,
    "cycling & jogging track": 8,
    "kids play pool with water slides": 8,
    "guest lobby in each floor": 8,
    "aesthetically designed landscape garden": 8,
    "health club with steam / jacuzzi": 8,
    "meditation area": 8,
    "pet park": 8,
    "visitor parking": 8,
    "badminton court": 8,
    "kids play area": 7,
    "community hall": 7,
    "power back up": 7,
    "cctv camera": 7,
    "rain water harvesting": 7,
    "internet/wi-fi connectivity": 7,
    "cycling track": 7,
    "art center": 7,
    "library": 7,
    "fire sprinklers": 7,
    "multipurpose hall": 7,
    "event space & amphitheatre": 7,
    "flower gardens": 6,
    "curated garden": 6,
    "multipurpose courts": 7,
    "dth television facility": 5,
    "fire fighting equipment": 6,
    "provision for power backup": 7,
    "sand pit": 6,
    "sewage treatment plant": 6,
    "solar energy": 7,
    "piped gas": 6,
    "kids club": 6,
    "waste disposal": 6,
    "lift": 5,
    "security": 5,
    "maintenance staff": 5,
    "reserved parking": 5,
    "ro water system": 5,
    "wheelchair accessibility": 5,
    "shopping center": 5,
    "laundry service": 5,
    "bank & atm": 5,
    "community entrance gate": 5,
    "canopy walk": 4,
    "entry exit gate": 4,
    "early learning centre": 4,
    "earth quake resistant": 7,
    "waste water recycling": 6,
    "whiteboard": 3,
    "printer": 3,
    "tea/coffee": 3,
    "house help accommodation": 7,
    "study room": 5,
    "ground water recharging": 5,
    "unknown": 0,
    "3 tier security system": 8,
    "ac in each room": 9,
    "activity deck4": 7,
    "aerobics room": 7,
    "air conditioned": 9,
    "all wooden flooring": 8,
    "arts & craft studio": 6,
    "bar/lounge": 7,
    "barbeque pit": 6,
    "barbeque space": 6,
    "cafeteria/food court": 7,
    "coffee lounge & restaurants": 7,
    "concierge services": 9,
    "conference room": 8,
    "cricket net practice": 6,
    "dance studio": 7,
    "downtown": 10,
    "fingerprint access": 8,
    "fireplace": 6,
    "golf course": 10,
    "hilltop": 10,
    "horticulture": 6,
    "indoor games room": 7,
    "island kitchen layout": 8,
    "jogging and strolling track": 7,
    "kids splash pool": 7,
    "lawn with pathway": 6,
    "guest accommodation":8,
    "marble flooring": 9,
    "mini cinema theatre": 9,
    "half basketball court":7,
    "park": 8,
    "pool with temperature control": 10,
    "intercom facility":6,
    "rentable community space": 6,
    "retail boulevard (retail shops)": 8,
    "service/goods lift": 6,
    "skydeck": 9,
    "vaastu compliant": 7,
    "volleyball court": 6,
    "water front": 10,
    "water storage": 5,
    "water treatment plant": 7,
    "wine cellar": 8
}

In [19]:
class AmenitiesScoreTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='amenities', weightages=None, output_column='assigned_amenities_score'):
        self.column = column
        self.weightages = weightages if weightages is not None else {}
        self.output_column = output_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        def calculate_score(amenities_str):
            # if not isinstance(amenities_str, str):
            #     return 0
            # amenities_list = [a.strip().lower() for a in amenities_str.split(",")]
            # return round(sum(self.weightages.get(a, 0) for a in amenities_list), 2)
            amenities_types = [f.strip().lower() for f in amenities_str.split(",")]
            total_weight = sum(self.weightages.get(f, 0) for f in amenities_types)
            return round(total_weight, 2)

        # Calculate scores
        X[self.output_column] = X[self.column].apply(calculate_score)

        # Replace 0 with NaN
        X[self.output_column] = X[self.output_column].replace(0, pd.NA)
        X[self.output_column] = pd.to_numeric(X[self.output_column], errors='coerce')

        # Drop original amenities column
        X.drop(columns=[self.column], inplace=True)

        return X


In [20]:
# Add missing indicator
class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column + '_missing'] = X[self.column].isna().astype(int)
        return X

In [21]:
# KNN imputation + MinMax scaling
class ImputeAndScaleAmenity(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score', n_neighbors=5):
        self.column = column
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        # Fit on the original column
        self.imputer.fit(X[[self.column]])
        imputed = self.imputer.transform(X[[self.column]])
        self.scaler.fit(imputed)
        return self

    def transform(self, X):
        X = X.copy()
        # Impute and scale the column
        imputed = self.imputer.transform(X[[self.column]])
        scaled = self.scaler.transform(imputed)
        # Replace with scaled
        X[self.column] = scaled
        return X

In [22]:
# Custom Ordinal Encoder Wrapper (for single column)
class ConstructionOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column='construction', categories=None):
        self.column = column
        self.categories = categories
        self.encoder = OrdinalEncoder(categories=self.categories, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, X, y=None):
        self.encoder.fit(X[[self.column]])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = self.encoder.transform(X[[self.column]])
        return X

'builder' - constant / top 200 / target encode   
'project_name' - constant / top 200 / target encode    
'location' - constant / top 200 / target encode    


'project_in_acres' - KNN Imputer    
'area' - KNN Imputer      
'education_mean_km' - KNN Imputer  
'education_min_km' - KNN Imputer  
'transport_mean_km' - KNN Imputer  
'transport_min_km' - KNN Imputer  
'shopping_centre_mean_km' - KNN Imputer  
'shopping_centre_min_km' - KNN Imputer  
'overall_min_mean_km' - KNN Imputer  
'overall_avg_mean_km' - KNN Imputer  
'overall_min_min_km' - KNN Imputer  
'overall_avg_min_km' - KNN Imputer  
'available_units' - KNN Imputer  
'towers' - KNN Imputer  
'flat_on_floor' - KNN Imputer  
'total_floor' - KNN Imputer  
'bath' - KNN Imputer  
'parking' - KNN Imputer  
'commercial_hub_mean_km' - KNN Imputer  
'commercial_hub_min_km' - KNN Imputer  
'balcony' - KNN Imputer  

'lattitude' - iterative imputer   
'longitude' - iterative imputer   

'lift' - median  

'property_type' - mode / OHE  
'status' - mode / ordinal encoding  
'furnish' - mode / ordinal encoding  


'ownership' - constant / OHE  
'facing' - constant / OHE  
'overlooking' - constant / multilable  
'extra_rooms' - constant / multilable  
'flooring' - constant / multilable  


'assigned_amenities_score' -  missingindicator then KNNimputation and then min_max_scale
'construction' -  missingindicator and ordinal_encode

'city' - OHE  
'seller' - OHE  


'education_within_2km' - MinMax Scaling    
'transport_within_2km' - MinMax Scaling     
'shopping_centre_within_2km' - MinMax Scaling    
'commercial_hub_within_2km' - MinMax Scaling  
'hospital_within_2km' - MinMax Scaling  
'tourist_within_2km' - MinMax Scaling  
'total_within_2km' - MinMax Scaling  

In [23]:
impute_topk_target_encoding_cols = ['builder', 'project_name', 'location']

features_to_fill_knn_standardization = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','total_floor','bath','parking',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
] #'costpersqft','emi'

features_to_fill_knn = ['flat_on_floor']

features_to_fill_iterative = ['lattitude','longitude']
features_to_fill_median = ['lift']

impute_mf_and_OHE = ['property_type']

impute_mf_and_ordinal_encode = ['status','furnish']
ordinal_categories = [
    ['under construction', 'ongoing', 'ready to move'],  # status
    ['unfurnished', 'semi-furnished', 'furnished']       # furnish
]

impute_missing_and_OHE = ['ownership', 'facing']

impute_missing_and_multilable = ['overlooking','extra_rooms','flooring']

assignweight_missingindicator_KNNimputation_minmaxscale = ['amenities']

missingindicator_ordinal_encode = ['construction']
construction_categories = [[
    'missing', 'under construction', 'new construction', 'less than 5 years',
    '5 to 10 years', '10 to 15 years', '15 to 20 years', 'above 20 years'
]]

onehotencode = ['city','seller']

min_max_scaling = ['education_within_2km','transport_within_2km','shopping_centre_within_2km',
                   'commercial_hub_within_2km','hospital_within_2km','tourist_within_2km','total_within_2km']

In [24]:
# --- Final Pipeline for Target Encoding Columns ---
builder_location_project_name_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("top_k", TopKCategoriesTransformer(top_k=200)),
    ("target_encoder", TargetEncoder()),
    ("scaler", StandardScaler())  
])

knn_then_standardize = Pipeline(steps=[
    ("knn_imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

iterative_then_standardize = Pipeline(steps=[
    ("iterative_imputer", IterativeImputer()),
    ("scaler", StandardScaler())
])

median_then_standardize = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

property_type_pipeline = Pipeline(steps=[
    ('impute' , SimpleImputer(strategy="most_frequent")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


status_furnish_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ('ordinal encode', OrdinalEncoder(categories=ordinal_categories,handle_unknown='use_encoded_value',unknown_value=-1 )),
    ('scaler', StandardScaler())
])


ownership_facing_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

overlooking_extra_rooms_flooring_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('multilable', MultiLabelBinarizerTransformer())
])

assigned_amenities_pipeline = Pipeline(steps=[
    ('calculate_score', AmenitiesScoreTransformer(
        column='amenities',
        weightages=amenities_weightages,
        output_column='assigned_amenities_score'
    )),
    ('add_missing_indicator', MissingIndicatorAdder(column='assigned_amenities_score')),
    ('impute_and_scale', ImputeAndScaleAmenity(column='assigned_amenities_score', n_neighbors=5))
])

construction_pipeline = Pipeline(steps=[
    ('add_missing_indicator', MissingIndicatorAdder(column='construction')),
    ('ordinal_encode', ConstructionOrdinalEncoder(column='construction', categories=construction_categories)),
    ('scaler', StandardScaler())
])

city_seller_pipeline = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

within2km_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# --- Unified Preprocessor ---
preprocessor = make_column_transformer(
    # impute_constant=missing, topK and Target Encoding
    (builder_location_project_name_pipeline, impute_topk_target_encoding_cols),

    # Imputation and standardization
    (knn_then_standardize, features_to_fill_knn_standardization),
    (iterative_then_standardize, features_to_fill_iterative),
    (median_then_standardize, features_to_fill_median),
    
    #imputattion
    (KNNImputer(n_neighbors=5), features_to_fill_knn),

    # impute = most_frequent and OHE 
    (property_type_pipeline, impute_mf_and_OHE),

    #impute = most_frequent and ordinal encoding
    (status_furnish_pipeline, impute_mf_and_ordinal_encode),

    #impute_constant=missing and OHE
    (ownership_facing_pipeline, impute_missing_and_OHE),

    #impute_constant=missing and multilable
    (overlooking_extra_rooms_flooring_pipeline, impute_missing_and_multilable),

    #missingindicator then KNNimputation and then min_max_scale
    (assigned_amenities_pipeline, assignweight_missingindicator_KNNimputation_minmaxscale),

    #missingindicator and ordinal_encode
    (construction_pipeline, missingindicator_ordinal_encode),

    # onehot_encoder
    (city_seller_pipeline, onehotencode),

    # MinMax Scaling
    (within2km_pipeline, min_max_scaling),

    # Keep other columns
    remainder='passthrough',
    verbose_feature_names_out=False,
    n_jobs=-1
)

# --- Final Pipeline ---
final_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor)
])

In [25]:
X_train_trans = final_pipeline.fit_transform(X_train,y_train_trans)
X_test_trans = final_pipeline.transform(X_test)

#print(X_train_trans.head())
#print(X_train_trans.isna().sum()[X_train_trans.isna().sum() > 0])


## feature selection using permutation importance

In [26]:
# 2. Train final model
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train_trans, y_train_trans)

from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd

# --- Get feature names from pipeline ---
try:
    # If your pipeline ends with a ColumnTransformer
    feature_names = final_pipeline.get_feature_names_out()
except AttributeError:
    # Fallback if using manual names
    feature_names = [f"f{i}" for i in range(X_train_trans.shape[1])]

# --- Calculate permutation importance ---
perm_result = permutation_importance(
    final_model,
    X_train_trans,
    y_train_trans,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)


# --- Sort by importance ---
sorted_idx = np.argsort(perm_result.importances_mean)[::-1]

print("=== Permutation Importance Rankings ===")
for idx in sorted_idx:
    print(f"{feature_names[idx]}: Mean Importance = {perm_result.importances_mean[idx]:.6f}, "
          f"Std = {perm_result.importances_std[idx]:.6f}")

# --- Identify low-importance features ---
threshold = 0.0001  # Adjust as needed
low_importance_indices = [idx for idx in sorted_idx if perm_result.importances_mean[idx] <= threshold]
low_importance_features = [feature_names[i] for i in low_importance_indices]

print("\n=== Features with Near-Zero Importance ===")
print(low_importance_features)

# Create new DataFrame with only low-importance features
low_importance_df = X_train_trans.iloc[:, low_importance_indices]
low_importance_df.columns = low_importance_features

print(f"\nShape of low-importance feature DataFrame: {low_importance_df.shape}")



=== Permutation Importance Rankings ===
f4: Mean Importance = 0.122833, Std = 0.001543
f82: Mean Importance = 0.099334, Std = 0.001574
f68: Mean Importance = 0.075633, Std = 0.001498
f18: Mean Importance = 0.048581, Std = 0.001077
f2: Mean Importance = 0.037062, Std = 0.000729
f24: Mean Importance = 0.022261, Std = 0.000495
f23: Mean Importance = 0.020505, Std = 0.000133
f17: Mean Importance = 0.009783, Std = 0.000166
f1: Mean Importance = 0.009264, Std = 0.000319
f19: Mean Importance = 0.009005, Std = 0.000188
f70: Mean Importance = 0.008810, Std = 0.000214
f0: Mean Importance = 0.007774, Std = 0.000236
f73: Mean Importance = 0.004593, Std = 0.000084
f20: Mean Importance = 0.004076, Std = 0.000073
f74: Mean Importance = 0.004043, Std = 0.000083
f26: Mean Importance = 0.003829, Std = 0.000067
f21: Mean Importance = 0.003810, Std = 0.000077
f81: Mean Importance = 0.003753, Std = 0.000084
f71: Mean Importance = 0.003699, Std = 0.000152
f69: Mean Importance = 0.003086, Std = 0.000138
f9: 

In [27]:
# === 1. Drop Low-Importance Features ===
# Drop by indices instead of names
X_train_final = X_train_trans.drop(X_train_trans.columns[low_importance_indices], axis=1)
X_test_final = X_test_trans.drop(X_test_trans.columns[low_importance_indices], axis=1)

print(f"Reduced features: {X_train_final.shape[1]} from {X_train_trans.shape[1]}")


# === 2. Retrain Final Model ===
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_train_final, y_train_trans)

# === 3. Predict (in transformed space) ===
y_pred_train_trans = final_model.predict(X_train_final)
y_pred_test_trans = final_model.predict(X_test_final)

# === 4. Clip predictions before inverse transform ===
min_val, max_val = y_train_trans.min(), y_train_trans.max()
y_pred_train = pt.inverse_transform(
    np.clip(y_pred_train_trans, min_val, max_val).reshape(-1, 1)
).ravel()
y_pred_test = pt.inverse_transform(
    np.clip(y_pred_test_trans, min_val, max_val).reshape(-1, 1)
).ravel()

# === 5. Define Metric Calculation Function ===
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

train_metrics = calc_metrics(y_train, y_pred_train)
test_metrics = calc_metrics(y_test, y_pred_test)

# === 6. Cross-validation Evaluation ===
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred))),
    'R2': make_scorer(r2_score)
}

cv_results = cross_validate(
    final_model,
    X_train_final,
    y_train_trans,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    n_jobs=-1
)

# === 7. Print Results Neatly ===
print("==== Train Metrics (Reduced Features) ====")
print(f"MAE: {train_metrics[0]:.4f} | MSE: {train_metrics[1]:.4f} | RMSE: {train_metrics[2]:.4f} | R²: {train_metrics[3]:.4f}")

print("\n==== Test Metrics (Reduced Features) ====")
print(f"MAE: {test_metrics[0]:.4f} | MSE: {test_metrics[1]:.4f} | RMSE: {test_metrics[2]:.4f} | R²: {test_metrics[3]:.4f}")

print("\n==== Cross-Validation Train Scores ====")
for key, value in cv_results.items():
    if key.startswith("train"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

print("\n==== Cross-Validation Test Scores ====")
for key, value in cv_results.items():
    if key.startswith("test"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")


Reduced features: 68 from 83
==== Train Metrics (Reduced Features) ====
MAE: 0.4708 | MSE: 3.1451 | RMSE: 1.7735 | R²: 0.8292

==== Test Metrics (Reduced Features) ====
MAE: 0.5471 | MSE: 3.1283 | RMSE: 1.7687 | R²: 0.7862

==== Cross-Validation Train Scores ====
train_MAE - Mean: 0.1401, Std: 0.0009
train_MSE - Mean: 0.0378, Std: 0.0004
train_RMSE - Mean: 0.1943, Std: 0.0010
train_R2 - Mean: 0.9622, Std: 0.0006

==== Cross-Validation Test Scores ====
test_MAE - Mean: 0.1925, Std: 0.0031
test_MSE - Mean: 0.0709, Std: 0.0024
test_RMSE - Mean: 0.2662, Std: 0.0046
test_R2 - Mean: 0.9290, Std: 0.0039


In [28]:
# # Plot RFECV Scores
# plt.figure(figsize=(10, 5))
# plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
#          rfecv.cv_results_['mean_test_score'])
# plt.xlabel("Number of Selected Features")
# plt.ylabel("Cross-Validation R² Score")
# plt.title("RFECV Feature Selection")
# plt.grid(True)
# plt.tight_layout()
# plt.show()


In [29]:
final_model.feature_importances_

array([0.02916096, 0.02329293, 0.07447277, 0.00266126, 0.15374031,
       0.00488542, 0.00397757, 0.00329223, 0.00292664, 0.00421387,
       0.00342055, 0.0044936 , 0.00396188, 0.00348446, 0.00454312,
       0.00296873, 0.00265256, 0.02969382, 0.14689472, 0.03109019,
       0.01153527, 0.0082982 , 0.00260515, 0.04241954, 0.04169931,
       0.0011124 , 0.00567845, 0.00040059, 0.00035787, 0.00048705,
       0.00132107, 0.00027603, 0.00037058, 0.00065253, 0.00033573,
       0.00089556, 0.00038984, 0.00048644, 0.00041594, 0.00035682,
       0.00174685, 0.00023979, 0.00057864, 0.00715032, 0.00031634,
       0.00064682, 0.00019401, 0.00151004, 0.00071718, 0.0005257 ,
       0.00187601, 0.00040353, 0.00267476, 0.00042866, 0.10395214,
       0.00601814, 0.01703489, 0.00797149, 0.00781418, 0.00582089,
       0.00282175, 0.00076378, 0.00244977, 0.00120599, 0.00069189,
       0.00061912, 0.00689733, 0.16100804])

In [30]:
# # feature importance plot

# (
#     pd.DataFrame(final_model.feature_importances_,
#              index=rfecv.transform(X_train_trans).columns,
#              columns=["importance"])
#     .sort_values(by="importance")
#     .plot(kind='barh',figsize=(10,10))
# )

In [31]:
# Unpack metrics
train_mae, train_mse, train_rmse, train_r2 = train_metrics
test_mae, test_mse, test_rmse, test_r2 = test_metrics

# Log experiment
with mlflow.start_run(run_name="Baseline_model with permutation importance and standardization"):
    # Log experiment type
    mlflow.log_param("experiment_type", "perstand")

    # Log model parameters
    mlflow.log_params(final_model.get_params())

    # Log train/test evaluation metrics
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    # Log mean cross-validation metrics
    mlflow.log_metric("cv_train_mae", np.mean(cv_results['train_MAE']))
    mlflow.log_metric("cv_train_mse", np.mean(cv_results['train_MSE']))
    mlflow.log_metric("cv_train_rmse", np.mean(cv_results['train_RMSE']))
    mlflow.log_metric("cv_train_r2", np.mean(cv_results['train_R2']))

    mlflow.log_metric("cv_val_mae", np.mean(cv_results['test_MAE']))
    mlflow.log_metric("cv_val_mse", np.mean(cv_results['test_MSE']))
    mlflow.log_metric("cv_val_rmse", np.mean(cv_results['test_RMSE']))
    mlflow.log_metric("cv_val_r2", np.mean(cv_results['test_R2']))

🏃 View run Baseline_model with permutation importance and standardization at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/8/runs/96b2416a2ae44db7a45cfc921f32a4d5
🧪 View experiment at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/8


# experiment :07 (Baseline_model + RFECV + standardization)

In [15]:
# mlflow experiment

mlflow.set_experiment("Exp 7 - Baseline model with RFECV and standardization")

2025/08/18 12:25:19 INFO mlflow.tracking.fluent: Experiment with name 'Exp 7 - Baseline model with RFECV and standardization' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/0b0401df62cb4d8f99aeca64d407551e', creation_time=1755500119543, experiment_id='9', last_update_time=1755500119543, lifecycle_stage='active', name='Exp 7 - Baseline model with RFECV and standardization', tags={}>

In [16]:
# --- Custom MultiLabel Binarizer ---
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.columns = []

    def fit(self, X, y=None):
        self.columns = X.columns
        for col in self.columns:
            mlb = MultiLabelBinarizer()
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            mlb.fit(split_data)
            self.encoders[col] = mlb
        return self

    def transform(self, X):
        output_parts = []
        for col in self.columns:
            mlb = self.encoders[col]
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            transformed = mlb.transform(split_data)
            col_names = [f"{col}_{cls}" for cls in mlb.classes_]
            output_parts.append(pd.DataFrame(transformed, columns=col_names, index=X.index))
        return pd.concat(output_parts, axis=1)

In [17]:
# --- Custom Transformer: Top K Categories ---
class TopKCategoriesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=200):
        self.top_k = top_k
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            top = X[col].value_counts().nlargest(self.top_k).index
            self.top_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].where(X[col].isin(self.top_categories_[col]), other='__other__')
        return X

In [18]:
#final
amenities_weightages = {
    "sea facing": 10,
    "private pool": 10,
    "private jaccuzi": 10,
    "sky villa": 10,
    "helipad": 10,
    "wrap around balcony": 7,
    "infinity swimming pool": 10,
    "high ceiling": 9,
    "located in the heart of city": 10,
    "large open space": 10,
    "skyline view": 10,
    "private terrace/garden": 10,
    "private garage": 10,
    "mansion": 10,
    "club house": 9,
    "large clubhouse": 9,
    "modular kitchen": 9,
    "central ac": 9,
    "banquet hall": 6,
    "premium branded fittings": 9,
    "private garden": 9,
    "full glass wall": 9,
    "garden view": 9,
    "theme based architectures": 9,
    "grand entrance lobby": 9,
    "smart home": 9,
    "library and business centre": 9,
    "recreational pool": 9,
    "projector": 8,
    "swimming pool": 8,
    "gymnasium": 8,
    "indoor squash & badminton courts": 8,
    "outdoor tennis courts": 8,
    "cycling & jogging track": 8,
    "kids play pool with water slides": 8,
    "guest lobby in each floor": 8,
    "aesthetically designed landscape garden": 8,
    "health club with steam / jacuzzi": 8,
    "meditation area": 8,
    "pet park": 8,
    "visitor parking": 8,
    "badminton court": 8,
    "kids play area": 7,
    "community hall": 7,
    "power back up": 7,
    "cctv camera": 7,
    "rain water harvesting": 7,
    "internet/wi-fi connectivity": 7,
    "cycling track": 7,
    "art center": 7,
    "library": 7,
    "fire sprinklers": 7,
    "multipurpose hall": 7,
    "event space & amphitheatre": 7,
    "flower gardens": 6,
    "curated garden": 6,
    "multipurpose courts": 7,
    "dth television facility": 5,
    "fire fighting equipment": 6,
    "provision for power backup": 7,
    "sand pit": 6,
    "sewage treatment plant": 6,
    "solar energy": 7,
    "piped gas": 6,
    "kids club": 6,
    "waste disposal": 6,
    "lift": 5,
    "security": 5,
    "maintenance staff": 5,
    "reserved parking": 5,
    "ro water system": 5,
    "wheelchair accessibility": 5,
    "shopping center": 5,
    "laundry service": 5,
    "bank & atm": 5,
    "community entrance gate": 5,
    "canopy walk": 4,
    "entry exit gate": 4,
    "early learning centre": 4,
    "earth quake resistant": 7,
    "waste water recycling": 6,
    "whiteboard": 3,
    "printer": 3,
    "tea/coffee": 3,
    "house help accommodation": 7,
    "study room": 5,
    "ground water recharging": 5,
    "unknown": 0,
    "3 tier security system": 8,
    "ac in each room": 9,
    "activity deck4": 7,
    "aerobics room": 7,
    "air conditioned": 9,
    "all wooden flooring": 8,
    "arts & craft studio": 6,
    "bar/lounge": 7,
    "barbeque pit": 6,
    "barbeque space": 6,
    "cafeteria/food court": 7,
    "coffee lounge & restaurants": 7,
    "concierge services": 9,
    "conference room": 8,
    "cricket net practice": 6,
    "dance studio": 7,
    "downtown": 10,
    "fingerprint access": 8,
    "fireplace": 6,
    "golf course": 10,
    "hilltop": 10,
    "horticulture": 6,
    "indoor games room": 7,
    "island kitchen layout": 8,
    "jogging and strolling track": 7,
    "kids splash pool": 7,
    "lawn with pathway": 6,
    "guest accommodation":8,
    "marble flooring": 9,
    "mini cinema theatre": 9,
    "half basketball court":7,
    "park": 8,
    "pool with temperature control": 10,
    "intercom facility":6,
    "rentable community space": 6,
    "retail boulevard (retail shops)": 8,
    "service/goods lift": 6,
    "skydeck": 9,
    "vaastu compliant": 7,
    "volleyball court": 6,
    "water front": 10,
    "water storage": 5,
    "water treatment plant": 7,
    "wine cellar": 8
}

In [19]:
class AmenitiesScoreTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='amenities', weightages=None, output_column='assigned_amenities_score'):
        self.column = column
        self.weightages = weightages if weightages is not None else {}
        self.output_column = output_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        def calculate_score(amenities_str):
            # if not isinstance(amenities_str, str):
            #     return 0
            # amenities_list = [a.strip().lower() for a in amenities_str.split(",")]
            # return round(sum(self.weightages.get(a, 0) for a in amenities_list), 2)
            amenities_types = [f.strip().lower() for f in amenities_str.split(",")]
            total_weight = sum(self.weightages.get(f, 0) for f in amenities_types)
            return round(total_weight, 2)

        # Calculate scores
        X[self.output_column] = X[self.column].apply(calculate_score)

        # Replace 0 with NaN
        X[self.output_column] = X[self.output_column].replace(0, pd.NA)
        X[self.output_column] = pd.to_numeric(X[self.output_column], errors='coerce')

        # Drop original amenities column
        X.drop(columns=[self.column], inplace=True)

        return X


In [20]:
# Add missing indicator
class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column + '_missing'] = X[self.column].isna().astype(int)
        return X

In [21]:
# KNN imputation + MinMax scaling
class ImputeAndScaleAmenity(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score', n_neighbors=5):
        self.column = column
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        # Fit on the original column
        self.imputer.fit(X[[self.column]])
        imputed = self.imputer.transform(X[[self.column]])
        self.scaler.fit(imputed)
        return self

    def transform(self, X):
        X = X.copy()
        # Impute and scale the column
        imputed = self.imputer.transform(X[[self.column]])
        scaled = self.scaler.transform(imputed)
        # Replace with scaled
        X[self.column] = scaled
        return X

In [22]:
# Custom Ordinal Encoder Wrapper (for single column)
class ConstructionOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column='construction', categories=None):
        self.column = column
        self.categories = categories
        self.encoder = OrdinalEncoder(categories=self.categories, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, X, y=None):
        self.encoder.fit(X[[self.column]])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = self.encoder.transform(X[[self.column]])
        return X

'builder' - constant / top 200 / target encode   
'project_name' - constant / top 200 / target encode    
'location' - constant / top 200 / target encode    


'project_in_acres' - KNN Imputer    
'area' - KNN Imputer      
'education_mean_km' - KNN Imputer  
'education_min_km' - KNN Imputer  
'transport_mean_km' - KNN Imputer  
'transport_min_km' - KNN Imputer  
'shopping_centre_mean_km' - KNN Imputer  
'shopping_centre_min_km' - KNN Imputer  
'overall_min_mean_km' - KNN Imputer  
'overall_avg_mean_km' - KNN Imputer  
'overall_min_min_km' - KNN Imputer  
'overall_avg_min_km' - KNN Imputer  
'available_units' - KNN Imputer  
'towers' - KNN Imputer  
'flat_on_floor' - KNN Imputer  
'total_floor' - KNN Imputer  
'bath' - KNN Imputer  
'parking' - KNN Imputer  
'commercial_hub_mean_km' - KNN Imputer  
'commercial_hub_min_km' - KNN Imputer  
'balcony' - KNN Imputer  

'lattitude' - iterative imputer   
'longitude' - iterative imputer   

'lift' - median  

'property_type' - mode / OHE  
'status' - mode / ordinal encoding  
'furnish' - mode / ordinal encoding  


'ownership' - constant / OHE  
'facing' - constant / OHE  
'overlooking' - constant / multilable  
'extra_rooms' - constant / multilable  
'flooring' - constant / multilable  


'assigned_amenities_score' -  missingindicator then KNNimputation and then min_max_scale
'construction' -  missingindicator and ordinal_encode

'city' - OHE  
'seller' - OHE  


'education_within_2km' - MinMax Scaling    
'transport_within_2km' - MinMax Scaling     
'shopping_centre_within_2km' - MinMax Scaling    
'commercial_hub_within_2km' - MinMax Scaling  
'hospital_within_2km' - MinMax Scaling  
'tourist_within_2km' - MinMax Scaling  
'total_within_2km' - MinMax Scaling  

In [23]:
impute_topk_target_encoding_cols = ['builder', 'project_name', 'location']

features_to_fill_knn_standardization = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','total_floor','bath','parking',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
] #'costpersqft','emi'

features_to_fill_knn = ['flat_on_floor']

features_to_fill_iterative = ['lattitude','longitude']
features_to_fill_median = ['lift']

impute_mf_and_OHE = ['property_type']

impute_mf_and_ordinal_encode = ['status','furnish']
ordinal_categories = [
    ['under construction', 'ongoing', 'ready to move'],  # status
    ['unfurnished', 'semi-furnished', 'furnished']       # furnish
]

impute_missing_and_OHE = ['ownership', 'facing']

impute_missing_and_multilable = ['overlooking','extra_rooms','flooring']

assignweight_missingindicator_KNNimputation_minmaxscale = ['amenities']

missingindicator_ordinal_encode = ['construction']
construction_categories = [[
    'missing', 'under construction', 'new construction', 'less than 5 years',
    '5 to 10 years', '10 to 15 years', '15 to 20 years', 'above 20 years'
]]

onehotencode = ['city','seller']

min_max_scaling = ['education_within_2km','transport_within_2km','shopping_centre_within_2km',
                   'commercial_hub_within_2km','hospital_within_2km','tourist_within_2km','total_within_2km']

In [24]:
# --- Final Pipeline for Target Encoding Columns ---
builder_location_project_name_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("top_k", TopKCategoriesTransformer(top_k=200)),
    ("target_encoder", TargetEncoder()),
    ("scaler", StandardScaler())  
])

knn_then_standardize = Pipeline(steps=[
    ("knn_imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

iterative_then_standardize = Pipeline(steps=[
    ("iterative_imputer", IterativeImputer()),
    ("scaler", StandardScaler())
])

median_then_standardize = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

property_type_pipeline = Pipeline(steps=[
    ('impute' , SimpleImputer(strategy="most_frequent")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


status_furnish_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ('ordinal encode', OrdinalEncoder(categories=ordinal_categories,handle_unknown='use_encoded_value',unknown_value=-1 )),
    ('scaler', StandardScaler())
])


ownership_facing_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

overlooking_extra_rooms_flooring_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('multilable', MultiLabelBinarizerTransformer())
])

assigned_amenities_pipeline = Pipeline(steps=[
    ('calculate_score', AmenitiesScoreTransformer(
        column='amenities',
        weightages=amenities_weightages,
        output_column='assigned_amenities_score'
    )),
    ('add_missing_indicator', MissingIndicatorAdder(column='assigned_amenities_score')),
    ('impute_and_scale', ImputeAndScaleAmenity(column='assigned_amenities_score', n_neighbors=5))
])

construction_pipeline = Pipeline(steps=[
    ('add_missing_indicator', MissingIndicatorAdder(column='construction')),
    ('ordinal_encode', ConstructionOrdinalEncoder(column='construction', categories=construction_categories)),
    ('scaler', StandardScaler())
])

city_seller_pipeline = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

within2km_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# --- Unified Preprocessor ---
preprocessor = make_column_transformer(
    # impute_constant=missing, topK and Target Encoding
    (builder_location_project_name_pipeline, impute_topk_target_encoding_cols),

    # Imputation and standardization
    (knn_then_standardize, features_to_fill_knn_standardization),
    (iterative_then_standardize, features_to_fill_iterative),
    (median_then_standardize, features_to_fill_median),
    
    #imputattion
    (KNNImputer(n_neighbors=5), features_to_fill_knn),

    # impute = most_frequent and OHE 
    (property_type_pipeline, impute_mf_and_OHE),

    #impute = most_frequent and ordinal encoding
    (status_furnish_pipeline, impute_mf_and_ordinal_encode),

    #impute_constant=missing and OHE
    (ownership_facing_pipeline, impute_missing_and_OHE),

    #impute_constant=missing and multilable
    (overlooking_extra_rooms_flooring_pipeline, impute_missing_and_multilable),

    #missingindicator then KNNimputation and then min_max_scale
    (assigned_amenities_pipeline, assignweight_missingindicator_KNNimputation_minmaxscale),

    #missingindicator and ordinal_encode
    (construction_pipeline, missingindicator_ordinal_encode),

    # onehot_encoder
    (city_seller_pipeline, onehotencode),

    # MinMax Scaling
    (within2km_pipeline, min_max_scaling),

    # Keep other columns
    remainder='passthrough',
    verbose_feature_names_out=False,
    n_jobs=-1
)

# --- Final Pipeline ---
final_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor)
])

In [25]:
X_train_trans = final_pipeline.fit_transform(X_train,y_train_trans)
X_test_trans = final_pipeline.transform(X_test)

#print(X_train_trans.head())
#print(X_train_trans.isna().sum()[X_train_trans.isna().sum() > 0])


In [26]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

## feature selection using RFECV

In [27]:
# Step Summary:
# 1. Feature selection using RFECV with RandomForestRegressor as the estimator.
# 2. Kept only the selected features from the training and test sets.
# 3. Trained a new RandomForestRegressor on the selected features.
# 4. Evaluated model performance using cross-validation and test metrics.

In [28]:
from sklearn.feature_selection import RFECV

In [29]:
# feature selection using rfecv

rfecv = RFECV(
    estimator=rf,
    step=1,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

In [30]:
# select features

rfecv.fit(X_train_trans, y_train_trans)
print("Selected features:", rfecv.n_features_)
print("Total features:", X_train_trans.shape[1])


Fitting estimator with 83 features.
Fitting estimator with 82 features.
Fitting estimator with 81 features.
Fitting estimator with 80 features.
Fitting estimator with 79 features.
Fitting estimator with 78 features.
Fitting estimator with 77 features.
Fitting estimator with 76 features.
Fitting estimator with 75 features.
Fitting estimator with 74 features.
Selected features: 73
Total features: 83


In [31]:
# Check which features were selected
rfecv.get_feature_names_out()

array(['builder', 'project_name', 'location', 'project_in_acres', 'area',
       'education_mean_km', 'education_min_km', 'transport_mean_km',
       'transport_min_km', 'shopping_centre_mean_km',
       'shopping_centre_min_km', 'overall_min_mean_km',
       'overall_avg_mean_km', 'overall_min_min_km', 'overall_avg_min_km',
       'available_units', 'towers', 'total_floor', 'bath', 'parking',
       'commercial_hub_mean_km', 'commercial_hub_min_km', 'balcony',
       'lattitude', 'longitude', 'lift', 'flat_on_floor',
       'property_type_new property', 'property_type_resale', 'status',
       'furnish', 'ownership_co-operative society', 'ownership_freehold',
       'ownership_missing', 'facing_east', 'facing_missing',
       'facing_north', 'facing_north - east', 'facing_south -west',
       'facing_west', 'overlooking_garden/park', 'overlooking_main road',
       'overlooking_missing', 'overlooking_pool', 'extra_rooms_missing',
       'extra_rooms_none of these', 'extra_rooms_puja',

In [32]:
# 1. Transform X data using selected features
X_train_selected = rfecv.transform(X_train_trans)
X_test_selected = rfecv.transform(X_test_trans)



# 2. Train final model
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train_selected, y_train_trans)


# 3. Predict (in transformed space)
y_pred_train_trans = final_model.predict(X_train_selected)
y_pred_test_trans = final_model.predict(X_test_selected)
print(y_pred_train_trans.shape)
print(y_pred_test_trans.shape)
print("="*50)

# 4. Clip predictions before inverse transform
min_val, max_val = y_train_trans.min(), y_train_trans.max()
y_pred_train = pt.inverse_transform(np.clip(y_pred_train_trans, min_val, max_val).reshape(-1, 1)).ravel()
y_pred_test = pt.inverse_transform(np.clip(y_pred_test_trans, min_val, max_val).reshape(-1, 1)).ravel()


# 5. Define metric calculation function
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

train_metrics = calc_metrics(y_train, y_pred_train)
test_metrics = calc_metrics(y_test, y_pred_test)


# 6. Cross-validation evaluation
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred))),
    'R2': make_scorer(r2_score)
}

cv_results = cross_validate(
    final_model,
    X_train_selected,
    y_train_trans,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    n_jobs=-1
)


# 7. Print results neatly
print("==== Train Metrics (Selected Features) ====")
print(f"MAE: {train_metrics[0]:.4f} | MSE: {train_metrics[1]:.4f} | RMSE: {train_metrics[2]:.4f} | R²: {train_metrics[3]:.4f}")

print("\n==== Test Metrics (Selected Features) ====")
print(f"MAE: {test_metrics[0]:.4f} | MSE: {test_metrics[1]:.4f} | RMSE: {test_metrics[2]:.4f} | R²: {test_metrics[3]:.4f}")

print("\n==== Cross-Validation Train Scores ====")
for key, value in cv_results.items():
    if key.startswith("train"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

print("\n==== Cross-Validation Test Scores ====")
for key, value in cv_results.items():
    if key.startswith("test"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")


(9302,)
(2326,)
==== Train Metrics (Selected Features) ====
MAE: 0.4889 | MSE: 3.4965 | RMSE: 1.8699 | R²: 0.8102

==== Test Metrics (Selected Features) ====
MAE: 0.5638 | MSE: 3.3487 | RMSE: 1.8299 | R²: 0.7711

==== Cross-Validation Train Scores ====
train_MAE - Mean: 0.1430, Std: 0.0010
train_MSE - Mean: 0.0392, Std: 0.0005
train_RMSE - Mean: 0.1980, Std: 0.0012
train_R2 - Mean: 0.9608, Std: 0.0007

==== Cross-Validation Test Scores ====
test_MAE - Mean: 0.1950, Std: 0.0028
test_MSE - Mean: 0.0728, Std: 0.0022
test_RMSE - Mean: 0.2697, Std: 0.0041
test_R2 - Mean: 0.9271, Std: 0.0036


In [33]:
# # Plot RFECV Scores
# plt.figure(figsize=(10, 5))
# plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
#          rfecv.cv_results_['mean_test_score'])
# plt.xlabel("Number of Selected Features")
# plt.ylabel("Cross-Validation R² Score")
# plt.title("RFECV Feature Selection")
# plt.grid(True)
# plt.tight_layout()
# plt.show()


In [34]:
final_model.feature_importances_

array([2.39163502e-02, 2.44707216e-02, 8.60481540e-02, 2.84445449e-03,
       1.77083327e-01, 4.03887800e-03, 4.32009516e-03, 3.69875348e-03,
       2.94781405e-03, 5.03317582e-03, 4.32180815e-03, 4.99079279e-03,
       4.02554613e-03, 4.67809367e-03, 4.90335667e-03, 2.94595958e-03,
       3.01127655e-03, 2.36916323e-02, 1.25058679e-01, 3.29887281e-02,
       1.14567867e-02, 1.00232288e-02, 3.33845474e-03, 3.97732392e-02,
       3.89379120e-02, 1.45479230e-03, 6.82064805e-03, 4.45402849e-04,
       5.96095605e-04, 5.74049552e-04, 1.38488398e-03, 2.57988877e-04,
       3.86329976e-04, 6.88949257e-04, 3.74563980e-04, 7.39688874e-04,
       4.13003594e-05, 1.26602864e-04, 7.12861365e-06, 9.73461932e-05,
       3.97221037e-04, 4.77062036e-04, 7.21628508e-04, 2.72910198e-04,
       4.04610829e-03, 2.57240967e-04, 6.09916050e-04, 9.91451228e-03,
       2.99531444e-04, 3.43290560e-04, 1.61776438e-04, 1.35733192e-04,
       1.99119552e-03, 6.83815744e-05, 1.14601813e-03, 4.90359815e-04,
      

In [35]:
# # feature importance plot

# (
#     pd.DataFrame(final_model.feature_importances_,
#              index=rfecv.transform(X_train_trans).columns,
#              columns=["importance"])
#     .sort_values(by="importance")
#     .plot(kind='barh',figsize=(10,10))
# )

In [36]:
# Unpack metrics
train_mae, train_mse, train_rmse, train_r2 = train_metrics
test_mae, test_mse, test_rmse, test_r2 = test_metrics

# Log experiment
with mlflow.start_run(run_name="Baseline model with RFECV and standardization"):
    # Log experiment type
    mlflow.log_param("experiment_type", "RFECVstand")

    # Log model parameters
    mlflow.log_params(final_model.get_params())

    # Log train/test evaluation metrics
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    # Log mean cross-validation metrics
    mlflow.log_metric("cv_train_mae", np.mean(cv_results['train_MAE']))
    mlflow.log_metric("cv_train_mse", np.mean(cv_results['train_MSE']))
    mlflow.log_metric("cv_train_rmse", np.mean(cv_results['train_RMSE']))
    mlflow.log_metric("cv_train_r2", np.mean(cv_results['train_R2']))

    mlflow.log_metric("cv_val_mae", np.mean(cv_results['test_MAE']))
    mlflow.log_metric("cv_val_mse", np.mean(cv_results['test_MSE']))
    mlflow.log_metric("cv_val_rmse", np.mean(cv_results['test_RMSE']))
    mlflow.log_metric("cv_val_r2", np.mean(cv_results['test_R2']))

🏃 View run Baseline model with RFECV and standardization at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/9/runs/2071cfe208704057bbf9930532b35ea3
🧪 View experiment at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/9


# experiment :08 (Baseline_model + simple LOF + Permutation Importance)

In [15]:
# mlflow experiment

mlflow.set_experiment("Exp 8 - Baseline model with LOF and Permutation Importance")

2025/08/19 07:59:38 INFO mlflow.tracking.fluent: Experiment with name 'Exp 8 - Baseline model with LOF and Permutation Importance' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/c7994a5364374f698cc35db0864f4ace', creation_time=1755570579052, experiment_id='10', last_update_time=1755570579052, lifecycle_stage='active', name='Exp 8 - Baseline model with LOF and Permutation Importance', tags={}>

In [16]:
# --- Custom MultiLabel Binarizer ---
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.columns = []

    def fit(self, X, y=None):
        self.columns = X.columns
        for col in self.columns:
            mlb = MultiLabelBinarizer()
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            mlb.fit(split_data)
            self.encoders[col] = mlb
        return self

    def transform(self, X):
        output_parts = []
        for col in self.columns:
            mlb = self.encoders[col]
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            transformed = mlb.transform(split_data)
            col_names = [f"{col}_{cls}" for cls in mlb.classes_]
            output_parts.append(pd.DataFrame(transformed, columns=col_names, index=X.index))
        return pd.concat(output_parts, axis=1)

In [17]:
# --- Custom Transformer: Top K Categories ---
class TopKCategoriesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=200):
        self.top_k = top_k
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            top = X[col].value_counts().nlargest(self.top_k).index
            self.top_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].where(X[col].isin(self.top_categories_[col]), other='__other__')
        return X

In [18]:
#final
amenities_weightages = {
    "sea facing": 10,
    "private pool": 10,
    "private jaccuzi": 10,
    "sky villa": 10,
    "helipad": 10,
    "wrap around balcony": 7,
    "infinity swimming pool": 10,
    "high ceiling": 9,
    "located in the heart of city": 10,
    "large open space": 10,
    "skyline view": 10,
    "private terrace/garden": 10,
    "private garage": 10,
    "mansion": 10,
    "club house": 9,
    "large clubhouse": 9,
    "modular kitchen": 9,
    "central ac": 9,
    "banquet hall": 6,
    "premium branded fittings": 9,
    "private garden": 9,
    "full glass wall": 9,
    "garden view": 9,
    "theme based architectures": 9,
    "grand entrance lobby": 9,
    "smart home": 9,
    "library and business centre": 9,
    "recreational pool": 9,
    "projector": 8,
    "swimming pool": 8,
    "gymnasium": 8,
    "indoor squash & badminton courts": 8,
    "outdoor tennis courts": 8,
    "cycling & jogging track": 8,
    "kids play pool with water slides": 8,
    "guest lobby in each floor": 8,
    "aesthetically designed landscape garden": 8,
    "health club with steam / jacuzzi": 8,
    "meditation area": 8,
    "pet park": 8,
    "visitor parking": 8,
    "badminton court": 8,
    "kids play area": 7,
    "community hall": 7,
    "power back up": 7,
    "cctv camera": 7,
    "rain water harvesting": 7,
    "internet/wi-fi connectivity": 7,
    "cycling track": 7,
    "art center": 7,
    "library": 7,
    "fire sprinklers": 7,
    "multipurpose hall": 7,
    "event space & amphitheatre": 7,
    "flower gardens": 6,
    "curated garden": 6,
    "multipurpose courts": 7,
    "dth television facility": 5,
    "fire fighting equipment": 6,
    "provision for power backup": 7,
    "sand pit": 6,
    "sewage treatment plant": 6,
    "solar energy": 7,
    "piped gas": 6,
    "kids club": 6,
    "waste disposal": 6,
    "lift": 5,
    "security": 5,
    "maintenance staff": 5,
    "reserved parking": 5,
    "ro water system": 5,
    "wheelchair accessibility": 5,
    "shopping center": 5,
    "laundry service": 5,
    "bank & atm": 5,
    "community entrance gate": 5,
    "canopy walk": 4,
    "entry exit gate": 4,
    "early learning centre": 4,
    "earth quake resistant": 7,
    "waste water recycling": 6,
    "whiteboard": 3,
    "printer": 3,
    "tea/coffee": 3,
    "house help accommodation": 7,
    "study room": 5,
    "ground water recharging": 5,
    "unknown": 0,
    "3 tier security system": 8,
    "ac in each room": 9,
    "activity deck4": 7,
    "aerobics room": 7,
    "air conditioned": 9,
    "all wooden flooring": 8,
    "arts & craft studio": 6,
    "bar/lounge": 7,
    "barbeque pit": 6,
    "barbeque space": 6,
    "cafeteria/food court": 7,
    "coffee lounge & restaurants": 7,
    "concierge services": 9,
    "conference room": 8,
    "cricket net practice": 6,
    "dance studio": 7,
    "downtown": 10,
    "fingerprint access": 8,
    "fireplace": 6,
    "golf course": 10,
    "hilltop": 10,
    "horticulture": 6,
    "indoor games room": 7,
    "island kitchen layout": 8,
    "jogging and strolling track": 7,
    "kids splash pool": 7,
    "lawn with pathway": 6,
    "guest accommodation":8,
    "marble flooring": 9,
    "mini cinema theatre": 9,
    "half basketball court":7,
    "park": 8,
    "pool with temperature control": 10,
    "intercom facility":6,
    "rentable community space": 6,
    "retail boulevard (retail shops)": 8,
    "service/goods lift": 6,
    "skydeck": 9,
    "vaastu compliant": 7,
    "volleyball court": 6,
    "water front": 10,
    "water storage": 5,
    "water treatment plant": 7,
    "wine cellar": 8
}

In [19]:
class AmenitiesScoreTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='amenities', weightages=None, output_column='assigned_amenities_score'):
        self.column = column
        self.weightages = weightages if weightages is not None else {}
        self.output_column = output_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        def calculate_score(amenities_str):
            # if not isinstance(amenities_str, str):
            #     return 0
            # amenities_list = [a.strip().lower() for a in amenities_str.split(",")]
            # return round(sum(self.weightages.get(a, 0) for a in amenities_list), 2)
            amenities_types = [f.strip().lower() for f in amenities_str.split(",")]
            total_weight = sum(self.weightages.get(f, 0) for f in amenities_types)
            return round(total_weight, 2)

        # Calculate scores
        X[self.output_column] = X[self.column].apply(calculate_score)

        # Replace 0 with NaN
        X[self.output_column] = X[self.output_column].replace(0, pd.NA)
        X[self.output_column] = pd.to_numeric(X[self.output_column], errors='coerce')

        # Drop original amenities column
        X.drop(columns=[self.column], inplace=True)

        return X


In [20]:
# Add missing indicator
class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column + '_missing'] = X[self.column].isna().astype(int)
        return X

In [21]:
# KNN imputation + MinMax scaling
class ImputeAndScaleAmenity(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score', n_neighbors=5):
        self.column = column
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        # Fit on the original column
        self.imputer.fit(X[[self.column]])
        imputed = self.imputer.transform(X[[self.column]])
        self.scaler.fit(imputed)
        return self

    def transform(self, X):
        X = X.copy()
        # Impute and scale the column
        imputed = self.imputer.transform(X[[self.column]])
        scaled = self.scaler.transform(imputed)
        # Replace with scaled
        X[self.column] = scaled
        return X

In [22]:
# Custom Ordinal Encoder Wrapper (for single column)
class ConstructionOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column='construction', categories=None):
        self.column = column
        self.categories = categories
        self.encoder = OrdinalEncoder(categories=self.categories, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, X, y=None):
        self.encoder.fit(X[[self.column]])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = self.encoder.transform(X[[self.column]])
        return X

'builder' - constant / top 200 / target encode   
'project_name' - constant / top 200 / target encode    
'location' - constant / top 200 / target encode    


'project_in_acres' - KNN Imputer    
'area' - KNN Imputer      
'education_mean_km' - KNN Imputer  
'education_min_km' - KNN Imputer  
'transport_mean_km' - KNN Imputer  
'transport_min_km' - KNN Imputer  
'shopping_centre_mean_km' - KNN Imputer  
'shopping_centre_min_km' - KNN Imputer  
'overall_min_mean_km' - KNN Imputer  
'overall_avg_mean_km' - KNN Imputer  
'overall_min_min_km' - KNN Imputer  
'overall_avg_min_km' - KNN Imputer  
'available_units' - KNN Imputer  
'towers' - KNN Imputer  
'flat_on_floor' - KNN Imputer  
'total_floor' - KNN Imputer  
'bath' - KNN Imputer  
'parking' - KNN Imputer  
'commercial_hub_mean_km' - KNN Imputer  
'commercial_hub_min_km' - KNN Imputer  
'balcony' - KNN Imputer  

'lattitude' - iterative imputer   
'longitude' - iterative imputer   

'lift' - median  

'property_type' - mode / OHE  
'status' - mode / ordinal encoding  
'furnish' - mode / ordinal encoding  


'ownership' - constant / OHE  
'facing' - constant / OHE  
'overlooking' - constant / multilable  
'extra_rooms' - constant / multilable  
'flooring' - constant / multilable  


'assigned_amenities_score' -  missingindicator then KNNimputation and then min_max_scale
'construction' -  missingindicator and ordinal_encode

'city' - OHE  
'seller' - OHE  


'education_within_2km' - MinMax Scaling    
'transport_within_2km' - MinMax Scaling     
'shopping_centre_within_2km' - MinMax Scaling    
'commercial_hub_within_2km' - MinMax Scaling  
'hospital_within_2km' - MinMax Scaling  
'tourist_within_2km' - MinMax Scaling  
'total_within_2km' - MinMax Scaling  

In [23]:
impute_topk_target_encoding_cols = ['builder', 'project_name', 'location']

features_to_fill_knn = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','flat_on_floor','total_floor','bath','parking',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
] #'costpersqft','emi'

features_to_fill_iterative = ['lattitude','longitude']
features_to_fill_median = ['lift']

impute_mf_and_OHE = ['property_type']

impute_mf_and_ordinal_encode = ['status','furnish']
ordinal_categories = [
    ['under construction', 'ongoing', 'ready to move'],  # status
    ['unfurnished', 'semi-furnished', 'furnished']       # furnish
]

impute_missing_and_OHE = ['ownership', 'facing']

impute_missing_and_multilable = ['overlooking','extra_rooms','flooring']

assignweight_missingindicator_KNNimputation_minmaxscale = ['amenities']

missingindicator_ordinal_encode = ['construction']
construction_categories = [[
    'missing', 'under construction', 'new construction', 'less than 5 years',
    '5 to 10 years', '10 to 15 years', '15 to 20 years', 'above 20 years'
]]

onehotencode = ['city','seller']

min_max_scaling = ['education_within_2km','transport_within_2km','shopping_centre_within_2km',
                   'commercial_hub_within_2km','hospital_within_2km','tourist_within_2km','total_within_2km']

In [24]:
print(X_train.shape)

(9302, 47)


In [25]:
# Here, only delete the rows that are outliers in the below columns.
# Even if imputation is applied below, it does NOT impute permanently —
# it only imputes temporarily to help detect and delete outlier data points.

features_to_fill_knn_OD  = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',                           
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','flat_on_floor','total_floor',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
]  #OD  means outlier detetion

features_to_fill_iterative_OD = ['lattitude','longitude']

# Impute OD columns separately for LOF
knn_imputer = KNNImputer(n_neighbors=5)
X_train_knn_imp = X_train[features_to_fill_knn_OD].copy()
X_train_knn_imp.loc[:, :] = knn_imputer.fit_transform(X_train_knn_imp)

iter_imputer = IterativeImputer()
X_train_iter_imp = X_train[features_to_fill_iterative_OD].copy()
X_train_iter_imp.loc[:, :] = iter_imputer.fit_transform(X_train_iter_imp)

# Apply LOF separately
lof_knn = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
lof_iter = LocalOutlierFactor(n_neighbors=20, contamination=0.01)

mask_knn = (lof_knn.fit_predict(X_train_knn_imp) == 1)
mask_iter = (lof_iter.fit_predict(X_train_iter_imp) == 1)

final_mask = mask_knn & mask_iter

# Filter train data and target
X_train_filtered = X_train.loc[final_mask].copy()
y_train_filtered = y_train_trans.loc[final_mask].copy()

print(f"Original train size: {X_train.shape[0]}")
print(f"Filtered train size after LOF: {X_train_filtered.shape[0]}")

Original train size: 9302
Filtered train size after LOF: 9114


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\neighbors\_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


In [26]:
# --- Final Pipeline for Target Encoding Columns ---
builder_location_project_name_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ('top_k', TopKCategoriesTransformer(top_k=200)),
    ('target_encoder', TargetEncoder())
])

property_type_pipeline = Pipeline(steps=[
    ('impute' , SimpleImputer(strategy="most_frequent")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


status_furnish_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ('ordinal encode', OrdinalEncoder(categories=ordinal_categories,handle_unknown='use_encoded_value',unknown_value=-1 ))
])


ownership_facing_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

overlooking_extra_rooms_flooring_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('multilable', MultiLabelBinarizerTransformer())
])

assigned_amenities_pipeline = Pipeline(steps=[
    ('calculate_score', AmenitiesScoreTransformer(
        column='amenities',
        weightages=amenities_weightages,
        output_column='assigned_amenities_score'
    )),
    ('add_missing_indicator', MissingIndicatorAdder(column='assigned_amenities_score')),
    ('impute_and_scale', ImputeAndScaleAmenity(column='assigned_amenities_score', n_neighbors=5))
])

construction_pipeline = Pipeline(steps=[
    ('add_missing_indicator', MissingIndicatorAdder(column='construction')),
    ('ordinal_encode', ConstructionOrdinalEncoder(column='construction', categories=construction_categories))
])

city_seller_pipeline = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

within2km_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# --- Unified Preprocessor ---
preprocessor = make_column_transformer(
    # impute_constant=missing, topK and Target Encoding
    (builder_location_project_name_pipeline, impute_topk_target_encoding_cols),

    # Imputation
    (KNNImputer(n_neighbors=5), features_to_fill_knn),
    (IterativeImputer(), features_to_fill_iterative),
    (SimpleImputer(strategy="median"), features_to_fill_median),

    # impute = most_frequent and OHE 
    (property_type_pipeline, impute_mf_and_OHE),

    #impute = most_frequent and ordinal encoding
    (status_furnish_pipeline, impute_mf_and_ordinal_encode),

    #impute_constant=missing and OHE
    (ownership_facing_pipeline, impute_missing_and_OHE),

    #impute_constant=missing and multilable
    (overlooking_extra_rooms_flooring_pipeline, impute_missing_and_multilable),

    #missingindicator then KNNimputation and then min_max_scale
    (assigned_amenities_pipeline, assignweight_missingindicator_KNNimputation_minmaxscale),

    #missingindicator and ordinal_encode
    (construction_pipeline, missingindicator_ordinal_encode),

    # onehot_encoder
    (city_seller_pipeline, onehotencode),

    # MinMax Scaling
    (within2km_pipeline, min_max_scaling),

    # Keep other columns
    remainder='passthrough',
    verbose_feature_names_out=False,
    n_jobs=-1
)

# --- Final Pipeline ---
final_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor)
])

In [27]:
X_train_trans = final_pipeline.fit_transform(X_train_filtered, y_train_filtered)
X_test_trans  = final_pipeline.transform(X_test)

#print(X_train_trans.head())
print(X_train_trans.isna().sum()[X_train_trans.isna().sum() > 0])

Series([], dtype: int64)


In [28]:
print(X_train_trans.shape)
print(y_train_filtered.shape)

(9114, 83)
(9114,)


## feature selection using permutation importance

In [29]:
# 2. Train final model
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train_trans, y_train_filtered)

from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd

# --- Get feature names from pipeline ---
try:
    # If your pipeline ends with a ColumnTransformer
    feature_names = final_pipeline.get_feature_names_out()
except AttributeError:
    # Fallback if using manual names
    feature_names = [f"f{i}" for i in range(X_train_trans.shape[1])]

# --- Calculate permutation importance ---
perm_result = permutation_importance(
    final_model,
    X_train_trans,
    y_train_filtered,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)


# --- Sort by importance ---
sorted_idx = np.argsort(perm_result.importances_mean)[::-1]

print("=== Permutation Importance Rankings ===")
for idx in sorted_idx:
    print(f"{feature_names[idx]}: Mean Importance = {perm_result.importances_mean[idx]:.6f}, "
          f"Std = {perm_result.importances_std[idx]:.6f}")

# --- Identify low-importance features ---
threshold = 0.0001  # Adjust as needed
low_importance_indices = [idx for idx in sorted_idx if perm_result.importances_mean[idx] <= threshold]
low_importance_features = [feature_names[i] for i in low_importance_indices]

print("\n=== Features with Near-Zero Importance ===")
print(low_importance_features)

# Create new DataFrame with only low-importance features
low_importance_df = X_train_trans.iloc[:, low_importance_indices]
low_importance_df.columns = low_importance_features

print(f"\nShape of low-importance feature DataFrame: {low_importance_df.shape}")



=== Permutation Importance Rankings ===
f4: Mean Importance = 0.120295, Std = 0.001291
f82: Mean Importance = 0.098595, Std = 0.001151
f68: Mean Importance = 0.067279, Std = 0.000972
f19: Mean Importance = 0.054958, Std = 0.000722
f2: Mean Importance = 0.038849, Std = 0.000559
f25: Mean Importance = 0.023500, Std = 0.000382
f24: Mean Importance = 0.023070, Std = 0.000673
f18: Mean Importance = 0.010468, Std = 0.000144
f1: Mean Importance = 0.009713, Std = 0.000305
f0: Mean Importance = 0.008620, Std = 0.000174
f70: Mean Importance = 0.007818, Std = 0.000326
f20: Mean Importance = 0.007583, Std = 0.000135
f73: Mean Importance = 0.004682, Std = 0.000175
f74: Mean Importance = 0.004278, Std = 0.000138
f71: Mean Importance = 0.004180, Std = 0.000164
f21: Mean Importance = 0.003893, Std = 0.000093
f22: Mean Importance = 0.003844, Std = 0.000081
f17: Mean Importance = 0.003809, Std = 0.000088
f69: Mean Importance = 0.003496, Std = 0.000142
f81: Mean Importance = 0.003490, Std = 0.000123
f11:

In [30]:
# === 1. Drop Low-Importance Features ===
# Drop by indices instead of names
X_train_final = X_train_trans.drop(X_train_trans.columns[low_importance_indices], axis=1)
X_test_final = X_test_trans.drop(X_test_trans.columns[low_importance_indices], axis=1)

print(f"Reduced features: {X_train_final.shape[1]} from {X_train_trans.shape[1]}")


# === 2. Retrain Final Model ===
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_train_final, y_train_filtered)

# === 3. Predict (in transformed space) ===
y_pred_train_trans = final_model.predict(X_train_final)
y_pred_test_trans = final_model.predict(X_test_final)

# === 4. Clip predictions before inverse transform ===
min_val, max_val = y_train_trans.min(), y_train_filtered.max()
y_pred_train = pt.inverse_transform(
    np.clip(y_pred_train_trans, min_val, max_val).reshape(-1, 1)
).ravel()
y_pred_test = pt.inverse_transform(
    np.clip(y_pred_test_trans, min_val, max_val).reshape(-1, 1)
).ravel()

y_train_filtered_inv = pt.inverse_transform(y_train_filtered.values.reshape(-1, 1)).ravel()

# === 5. Define Metric Calculation Function ===
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

train_metrics = calc_metrics(y_train_filtered_inv, y_pred_train)
test_metrics = calc_metrics(y_test, y_pred_test)

# === 6. Cross-validation Evaluation ===
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred))),
    'R2': make_scorer(r2_score)
}

cv_results = cross_validate(
    final_model,
    X_train_final,
    y_train_filtered,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    n_jobs=-1
)

# === 7. Print Results Neatly ===
print("==== Train Metrics (Reduced Features) ====")
print(f"MAE: {train_metrics[0]:.4f} | MSE: {train_metrics[1]:.4f} | RMSE: {train_metrics[2]:.4f} | R²: {train_metrics[3]:.4f}")

print("\n==== Test Metrics (Reduced Features) ====")
print(f"MAE: {test_metrics[0]:.4f} | MSE: {test_metrics[1]:.4f} | RMSE: {test_metrics[2]:.4f} | R²: {test_metrics[3]:.4f}")

print("\n==== Cross-Validation Train Scores ====")
for key, value in cv_results.items():
    if key.startswith("train"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

print("\n==== Cross-Validation Test Scores ====")
for key, value in cv_results.items():
    if key.startswith("test"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")


Reduced features: 68 from 83
==== Train Metrics (Reduced Features) ====
MAE: 0.4567 | MSE: 2.5374 | RMSE: 1.5929 | R²: 0.8470

==== Test Metrics (Reduced Features) ====
MAE: 0.5564 | MSE: 3.1987 | RMSE: 1.7885 | R²: 0.7814

==== Cross-Validation Train Scores ====
train_MAE - Mean: 0.1399, Std: 0.0006
train_MSE - Mean: 0.0376, Std: 0.0004
train_RMSE - Mean: 0.1938, Std: 0.0011
train_R2 - Mean: 0.9619, Std: 0.0007

==== Cross-Validation Test Scores ====
test_MAE - Mean: 0.1926, Std: 0.0033
test_MSE - Mean: 0.0712, Std: 0.0027
test_RMSE - Mean: 0.2668, Std: 0.0052
test_R2 - Mean: 0.9276, Std: 0.0047


In [31]:
# # Plot RFECV Scores
# plt.figure(figsize=(10, 5))
# plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
#          rfecv.cv_results_['mean_test_score'])
# plt.xlabel("Number of Selected Features")
# plt.ylabel("Cross-Validation R² Score")
# plt.title("RFECV Feature Selection")
# plt.grid(True)
# plt.tight_layout()
# plt.show()


In [32]:
final_model.feature_importances_

array([2.92806618e-02, 2.52543877e-02, 7.81880972e-02, 2.70616472e-03,
       1.64246932e-01, 3.99834922e-03, 3.29024349e-03, 3.03077444e-03,
       3.18669512e-03, 4.22902320e-03, 4.16684930e-03, 4.23170687e-03,
       3.92018045e-03, 3.92003579e-03, 4.43991963e-03, 2.99453920e-03,
       2.98601885e-03, 8.50789386e-03, 2.64405848e-02, 1.22020833e-01,
       3.57909556e-02, 8.81392829e-03, 7.20660736e-03, 2.20720302e-03,
       3.87760755e-02, 3.81009919e-02, 1.19247812e-03, 4.28214848e-04,
       6.36242608e-04, 5.23191613e-04, 1.42385460e-03, 2.29004282e-04,
       3.72717407e-04, 4.63364842e-04, 3.52577654e-04, 7.08703851e-04,
       4.09242482e-04, 4.25999100e-04, 5.48713784e-04, 2.69501520e-04,
       2.82841930e-03, 2.20740152e-04, 3.43586500e-04, 8.28165786e-03,
       4.32977687e-04, 3.27593854e-04, 1.51233432e-04, 1.60960473e-03,
       1.18779356e-03, 5.04461704e-04, 1.78226586e-03, 5.32731444e-04,
       2.65699850e-03, 4.81879434e-04, 9.30406497e-02, 6.44266510e-03,
      

In [33]:
# # feature importance plot

# (
#     pd.DataFrame(final_model.feature_importances_,
#              index=rfecv.transform(X_train_trans).columns,
#              columns=["importance"])
#     .sort_values(by="importance")
#     .plot(kind='barh',figsize=(10,10))
# )

In [34]:
# Unpack metrics
train_mae, train_mse, train_rmse, train_r2 = train_metrics
test_mae, test_mse, test_rmse, test_r2 = test_metrics

# Log experiment
with mlflow.start_run(run_name="Baseline model with LOF and Permutation Importance"):
    # Log experiment type
    mlflow.log_param("experiment_type", "LOFPer")

    # Log model parameters
    mlflow.log_params(final_model.get_params())

    # Log train/test evaluation metrics
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    # Log mean cross-validation metrics
    mlflow.log_metric("cv_train_mae", np.mean(cv_results['train_MAE']))
    mlflow.log_metric("cv_train_mse", np.mean(cv_results['train_MSE']))
    mlflow.log_metric("cv_train_rmse", np.mean(cv_results['train_RMSE']))
    mlflow.log_metric("cv_train_r2", np.mean(cv_results['train_R2']))

    mlflow.log_metric("cv_val_mae", np.mean(cv_results['test_MAE']))
    mlflow.log_metric("cv_val_mse", np.mean(cv_results['test_MSE']))
    mlflow.log_metric("cv_val_rmse", np.mean(cv_results['test_RMSE']))
    mlflow.log_metric("cv_val_r2", np.mean(cv_results['test_R2']))

🏃 View run Baseline model with LOF and Permutation Importance at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/10/runs/b224def3d8ec44aeb7a72287cc91dab2
🧪 View experiment at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/10


`observation`  
-1. Baseline + Permutation Importance (no Std) → Best Test R² (0.7911)  
-2. Baseline + RFECV → Close second, balanced performance  
-3. LOF + Permutation Importance → overfits slightly (good train, weaker test)  
-4. Baseline + Permutation + Standardization → Slightly lower Test R² but best CV R²    
-5. Baseline model with RFECV and standardization → Clearly weakest (lowest Test & CV R²)  